<a href="https://colab.research.google.com/github/Podrimate/THz_sim_application/blob/main/notebooks/THzSim_User_Workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# THzSim User Workflow

Use this notebook to build an arbitrary sample, define a measurement setup, and run synthetic parameter studies.

This version focuses on ease of use:
- common measurement/study settings are Colab form fields
- arbitrary samples and truth sweeps are edited as validated JSON blocks
- measured reference upload behaves like the fit notebook
- the notebook exports both `study_setup.json` and the legacy CSV setup file


## 0. Install And Import

Run the next cell first. In Colab it installs the package from GitHub automatically.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PACKAGE_IMPORT_OK = False
IMPORT_EXCEPTION = None
DEFAULT_PIP_SPEC = 'https://github.com/Podrimate/THz_sim_application/archive/refs/heads/main.zip'

def try_import_runtime_dependencies():
    global IMPORT_EXCEPTION
    try:
        import numpy  # noqa: F401
        import scipy  # noqa: F401
        import matplotlib  # noqa: F401
        return True
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return False

def try_import_thzsim2():
    global IMPORT_EXCEPTION
    try:
        importlib.invalidate_caches()
        import thzsim2  # noqa: F401
        return thzsim2
    except Exception as exc:
        IMPORT_EXCEPTION = exc
        return None

if IN_COLAB:
    pip_spec = os.environ.get('THZSIM_PIP_SPEC', DEFAULT_PIP_SPEC).strip()
    subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'thzsim2'])
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--force-reinstall',
        '--no-cache-dir',
        '--no-deps',
        pip_spec,
    ])
    sys.modules.pop('thzsim2', None)
    sys.modules.pop('thzsim2.notebook_api', None)

if not try_import_runtime_dependencies():
    raise RuntimeError(
        'The Python runtime dependencies are currently broken. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime, then rerun the first cell. '
        "The notebook now installs only thzsim2 and leaves Colab's NumPy/SciPy/Matplotlib in place."
    )

thzsim2 = try_import_thzsim2()
PACKAGE_IMPORT_OK = thzsim2 is not None

if not PACKAGE_IMPORT_OK:
    for candidate in [Path.cwd(), Path.cwd().parent]:
        if (candidate / 'thzsim2').exists():
            sys.path.insert(0, str(candidate))
            thzsim2 = try_import_thzsim2()
            if thzsim2 is not None:
                PACKAGE_IMPORT_OK = True
                break

if not PACKAGE_IMPORT_OK:
    raise RuntimeError(
        'Could not import thzsim2. '
        f'Last import error: {IMPORT_EXCEPTION!r}. '
        'In Colab use Runtime -> Factory reset runtime and rerun the first cell; '
        'locally open the notebook inside the repo.'
    )

import matplotlib
import numpy
import scipy

print(f'Using thzsim2 from: {thzsim2.__file__}')
print(f'thzsim2 version: {getattr(thzsim2, "__version__", "unknown")}')
print(f'numpy version: {numpy.__version__}')
print(f'scipy version: {scipy.__version__}')
print(f'matplotlib version: {matplotlib.__version__}')


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

from thzsim2.notebook_api import (
    build_study_setup,
    generate_reference_pulse,
    load_reference_csv,
    load_study_setup_json,
    run_study_from_setup_json,
    show_study_heatmaps,
    summarize_trace_input,
    write_study_setup_csv,
    write_study_setup_json,
)

workflow_root = Path.cwd() / 'thz_sim_workflow_outputs'
uploads_root = workflow_root / 'uploads'
workflow_root.mkdir(parents=True, exist_ok=True)
uploads_root.mkdir(parents=True, exist_ok=True)

def find_repo_path(relative_path):
    for candidate in [Path.cwd(), Path.cwd().parent]:
        candidate_path = candidate / relative_path
        if candidate_path.exists():
            return candidate_path.resolve()
    return None

LOCAL_TEST_DATA = find_repo_path('Test_data_for_fitter')

def upload_single_csv(target_dir, *, prompt):
    print(prompt)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError('Colab upload is only available inside Google Colab.') from exc
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    name, payload = next(iter(uploaded.items()))
    path = Path(target_dir) / name
    path.write_bytes(payload)
    return path.resolve()

def parse_json_block(text, *, label):
    try:
        return json.loads(text)
    except json.JSONDecodeError as exc:
        raise ValueError(f'{label} JSON is invalid at line {exc.lineno}, column {exc.colno}: {exc.msg}') from exc

def show_mapping(title, mapping):
    display(Markdown(f'**{title}**'))
    for key, value in mapping.items():
        print(f'- {key}: {value}')


## 1. Measurement And Reference Setup

What these parameters mean:
- `use_measured_reference`: use a CSV measured without the sample in place, from the same setup.
- `measurement_mode`: choose `transmission` or `reflection`.
- `angle_deg`: effective angle of incidence at the sample.
- `polarization_model`: choose `s`, `p`, or `mixed`.
- `reference_standard_kind='identity'`: compare the sample directly to the uploaded/generated reference.
- `reference_standard_kind='stack'`: divide by a modeled standard stack first.

Safe starting point:
- use `identity` for the reference standard unless you explicitly measured against a known standard stack.


In [ ]:
use_measured_reference = False #@param {type:"boolean"}
use_colab_upload_for_reference = False #@param {type:"boolean"}
reference_csv_path = "" #@param {type:"string"}
measurement_mode = "transmission" #@param ["transmission", "reflection"]
angle_deg = 0.0 #@param {type:"number"}
polarization_model = "mixed" #@param ["s", "p", "mixed"]
polarization_mix = 0.5 #@param {type:"number"}
reference_standard_kind = "identity" #@param ["identity", "stack"]

generated_model = "sech_carrier" #@param ["sech_carrier", "gaussian_carrier"]
generated_sample_count = 512 #@param {type:"integer"}
generated_dt_ps = 0.03 #@param {type:"number"}
generated_time_center_ps = 8.0 #@param {type:"number"}
generated_pulse_center_ps = 5.0 #@param {type:"number"}
generated_tau_ps = 0.22 #@param {type:"number"}
generated_f0_thz = 0.9 #@param {type:"number"}
generated_amp = 1.0 #@param {type:"number"}
generated_phi_rad = 0.0 #@param {type:"number"}


### Optional Reference Standard Stack JSON

Only edit the next cell if `reference_standard_kind='stack'`.


In [ ]:
reference_standard_stack_json = r'''
{
  "layers": []
}
'''


In [ ]:
selected_reference_csv = Path(reference_csv_path).expanduser().resolve() if reference_csv_path.strip() else None
if use_measured_reference and use_colab_upload_for_reference:
    selected_reference_csv = upload_single_csv(uploads_root, prompt='Upload the measured reference CSV file.')

if use_measured_reference:
    if selected_reference_csv is None:
        raise RuntimeError('Set reference_csv_path or enable Colab upload when use_measured_reference=True.')
    reference_import_summary = summarize_trace_input(selected_reference_csv)
    show_mapping('Measured Reference Import Summary', reference_import_summary)
    measured_reference = load_reference_csv(selected_reference_csv)
    plt.figure(figsize=(10, 4))
    plt.plot(measured_reference.time_ps, measured_reference.trace, label='measured reference')
    plt.title('Measured Reference Preview')
    plt.xlabel('Time (ps)')
    plt.ylabel('Signal')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    reference_config = {
        'kind': 'measured_csv',
        'path': selected_reference_csv,
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-study-measured-reference',
        },
    }
else:
    generated_reference_input = generate_reference_pulse(
        model=generated_model,
        sample_count=generated_sample_count,
        dt_ps=generated_dt_ps,
        time_center_ps=generated_time_center_ps,
        pulse_center_ps=generated_pulse_center_ps,
        tau_ps=generated_tau_ps,
        f0_thz=generated_f0_thz,
        amp=generated_amp,
        phi_rad=generated_phi_rad,
    )
    plt.figure(figsize=(10, 4))
    plt.plot(generated_reference_input.time_ps, generated_reference_input.trace, label='generated reference')
    plt.title('Generated Reference Preview')
    plt.xlabel('Time (ps)')
    plt.ylabel('Signal')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    reference_config = {
        'kind': 'generated_pulse',
        'generate': {
            'model': generated_model,
            'sample_count': generated_sample_count,
            'dt_ps': generated_dt_ps,
            'time_center_ps': generated_time_center_ps,
            'pulse_center_ps': generated_pulse_center_ps,
            'tau_ps': generated_tau_ps,
            'f0_thz': generated_f0_thz,
            'amp': generated_amp,
            'phi_rad': generated_phi_rad,
        },
        'prepare': {
            'output_root': workflow_root / 'runs',
            'run_label': 'user-study-generated-reference',
        },
    }


## 2. Sample Definition And Study Sweep

Edit the next two JSON blocks.

What to edit:
- `sample_layers_json`: the stack you want to simulate and fit
- `study_truth_json`: which parameters are true fixed values and which are sweep axes

How to encode fitable sample parameters:
- use the same `Fit` blocks as in the fit notebook
- any study truth entry with a list becomes a sweep axis
- any study truth entry with a scalar is kept fixed


In [ ]:
sample_n_in = 1.0 #@param {type:"number"}
sample_n_out = 1.0 #@param {type:"number"}
overlay_imported_nk = True #@param {type:"boolean"}
noise_dynamic_range_db = 80.0 #@param {type:"number"}
replicates = 1 #@param {type:"integer"}
study_seed = 5 #@param {type:"integer"}
study_metric = "mse" #@param ["mse", "relative_l2"]
max_internal_reflections = 2 #@param {type:"integer"}
optimizer_maxiter = 12 #@param {type:"integer"}
optimizer_global_maxiter = 1 #@param {type:"integer"}
optimizer_popsize = 5 #@param {type:"integer"}


In [ ]:
sample_layers_json = r'''
[
  {
    "name": "drude_layer",
    "thickness_um": {
      "kind": "Fit",
      "initial": 550.0,
      "abs_min": 300.0,
      "abs_max": 800.0,
      "label": "film_thickness_um"
    },
    "material": {
      "kind": "Drude",
      "eps_inf": 12.0,
      "plasma_freq_thz": {
        "kind": "Fit",
        "initial": 1.1,
        "abs_min": 0.1,
        "abs_max": 3.0,
        "label": "film_plasma_freq_thz"
      },
      "gamma_thz": {
        "kind": "Fit",
        "initial": 0.08,
        "abs_min": 0.005,
        "abs_max": 0.5,
        "label": "film_gamma_thz"
      }
    }
  }
]
'''

study_truth_json = r'''
{
  "layers[0].thickness_um": [500.0, 550.0, 600.0],
  "layers[0].material.plasma_freq_thz": [0.9, 1.1, 1.3],
  "layers[0].material.gamma_thz": 0.08
}
'''


## 3. Export The Study Setup

The notebook writes both a JSON setup file and the older CSV setup file.

Use the JSON file as the main review/share file.


In [ ]:
study_setup_json_path = str((workflow_root / 'study_setup.json').resolve()) #@param {type:"string"}
study_setup_csv_path = str((workflow_root / 'study_setup.csv').resolve()) #@param {type:"string"}
study_output_dir = str((workflow_root / 'study_run').resolve()) #@param {type:"string"}


In [ ]:
sample_layers = parse_json_block(sample_layers_json, label='Sample layers')
study_truth = parse_json_block(study_truth_json, label='Study truth')
reference_standard_stack = parse_json_block(reference_standard_stack_json, label='Reference standard stack')

measurement_config = {
    'mode': measurement_mode,
    'angle_deg': angle_deg,
    'polarization': polarization_model,
    'polarization_mix': polarization_mix if polarization_model == 'mixed' else None,
    'reference_standard': {'kind': 'identity'} if reference_standard_kind == 'identity' else {'kind': 'stack', 'stack': reference_standard_stack},
}

study_setup = build_study_setup(
    reference=reference_config,
    layers=sample_layers,
    measurement=measurement_config,
    study={
        'truth': study_truth,
        'noise_dynamic_range_db': noise_dynamic_range_db,
        'replicates': replicates,
        'seed': study_seed,
        'metric': study_metric,
        'max_internal_reflections': max_internal_reflections,
        'optimizer': {
            'method': 'L-BFGS-B',
            'options': {'maxiter': int(optimizer_maxiter)},
            'global_options': {'maxiter': int(optimizer_global_maxiter), 'popsize': int(optimizer_popsize), 'seed': int(study_seed)},
            'fd_rel_step': 1e-4,
        },
        'out_dir': study_output_dir,
    },
    n_in=sample_n_in,
    n_out=sample_n_out,
    overlay_imported=overlay_imported_nk,
)
saved_json_path = write_study_setup_json(study_setup_json_path, study_setup)
saved_csv_path = write_study_setup_csv(study_setup_csv_path, study_setup)
show_mapping('Saved Study Setup Files', {'study_setup_json': saved_json_path, 'study_setup_csv': saved_csv_path})


## 4. Run The Study And Review The Heatmaps

What you should see:
- the summary rows contain one row per simulated case/replicate
- `normalized_mse` and `relative_l2` heatmaps show which truth combinations are hardest to recover
- `signed-err__...` heatmaps show where fitted parameters are biased high or low


In [ ]:
loaded_study_setup = load_study_setup_json(saved_json_path)
show_mapping('Loaded Study Setup Sections', {key: type(value).__name__ for key, value in loaded_study_setup.items()})
study_result = run_study_from_setup_json(saved_json_path)
print('Study output directory:', study_result.out_dir)
print('Number of summary rows:', len(study_result.summary_rows))
study_result.summary_rows[:3]


In [ ]:
show_study_heatmaps(study_result, contains='normalized-mse__', max_images=6)
show_study_heatmaps(study_result, contains='relative-l2__', max_images=6)
show_study_heatmaps(study_result, contains='signed-err', max_images=9)


## 5. Files To Keep Or Send Back

The most useful review file is `study_setup.json`.

If you want me to inspect the study later, send:
- `study_setup.json`
- optionally the generated `study_summary.csv`
- optionally screenshots of the heatmaps that look suspicious or interesting
